## 1. Before You Begin

MAI-Image-2.5 is a next-generation image model that supports:
- **Text-to-image generation** — produce a PNG from a text prompt
- **Image-to-image editing** — modify an existing PNG with a targeted instruction

**Pricing:** $0.05 / image — [current rates](https://microsoft.ai/models/mai-image-2-5/)

**Prerequisites**

1. A Microsoft Foundry project with MAI-Image-2.5 deployed.
   See [models/quickstart/](../../quickstart/README.md) for first-time setup.
2. The three environment variables listed in section 2 set in your shell or `.env` file.
3. `requests` installed: `pip install requests`

**API constraints** ([source: Learn docs](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/how-to/use-foundry-models-mai-image?tabs=python))

| Rule | Value |
|:---|:---|
| Minimum dimension (each) | 768 px |
| Maximum total pixels | 1,048,576 (≈ 1024 × 1024) |
| `size` format (edits endpoint) | `"WxH"` string, e.g. `"1024x1024"` |
| `width` / `height` (generations endpoint) | Separate integers |
| Output format | PNG only |
| Auth header | `api-key` |


## 2. Set up your environment

The cell below checks all required variables and derives `BASE_ENDPOINT` from
`MICROSOFT_FOUNDRY_ENDPOINT`.

`MICROSOFT_FOUNDRY_ENDPOINT` has the form:
```
https://<resource>.services.ai.azure.com/api/projects/<project>
```
Stripping the `/api/projects/...` path gives the base URL the MAI image APIs use.

> **Deployment names** — `scripts/sample.env` defaults each deployment name to the
> model name (e.g. `AZURE_MAI_IMAGE_25_DEPLOYMENT=mai-image-2.5`).
> If your Foundry deployment uses a different name, update your `.env` file and
> re-run `scripts/setenv.sh` to refresh it before continuing.


In [ ]:
%pip install requests python-dotenv --quiet

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # loads .env from the current directory or any parent

In [ ]:
import os, base64
import requests
from pathlib import Path
from urllib.parse import urlparse
from IPython.display import Image as IPyImage, display

REQUIRED = {
    "MICROSOFT_FOUNDRY_ENDPOINT":         "Foundry project endpoint",
    "MICROSOFT_FOUNDRY_API_KEY":           "AIServices API key",
    "AZURE_MAI_IMAGE_25_DEPLOYMENT":       "MAI-Image-2.5 deployment name",
}
missing = [k for k in REQUIRED if not os.environ.get(k)]
if missing:
    raise EnvironmentError(f"Set these env vars before continuing: {missing}")

parsed        = urlparse(os.environ["MICROSOFT_FOUNDRY_ENDPOINT"])
BASE_ENDPOINT = f"{parsed.scheme}://{parsed.netloc}"
API_KEY       = os.environ["MICROSOFT_FOUNDRY_API_KEY"]
DEPLOYMENT    = os.environ["AZURE_MAI_IMAGE_25_DEPLOYMENT"]

print(f"Base endpoint : {BASE_ENDPOINT}")
print(f"Deployment    : {DEPLOYMENT}")
print("Environment check passed.")


## 3. Configure the client

`generate_image` and `edit_image` wrap the two API endpoints.
Both validate the pixel budget before making a network call so errors surface locally.

`show` decodes the `data[0].b64_json` field in the response and renders it inline.


In [ ]:
GEN_URL  = f"{BASE_ENDPOINT}/mai/v1/images/generations"
EDIT_URL = f"{BASE_ENDPOINT}/mai/v1/images/edits"

OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)


def generate_image(prompt: str, width: int = 1024, height: int = 1024) -> str:
    """POST to /mai/v1/images/generations; return base64 PNG string."""
    assert width >= 768 and height >= 768, "Each dimension must be ≥ 768 px"
    assert width * height <= 1_048_576,    "width × height must be ≤ 1,048,576"
    resp = requests.post(
        GEN_URL,
        headers={"Content-Type": "application/json", "api-key": API_KEY},
        json={"model": DEPLOYMENT, "prompt": prompt, "width": width, "height": height},
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()["data"][0]["b64_json"]


def edit_image(image_path: str, prompt: str, size: str = "1024x1024") -> str:
    """POST to /mai/v1/images/edits (multipart); return base64 PNG string."""
    w, h = (int(d) for d in size.split("x"))
    assert w >= 768 and h >= 768, "Each dimension must be ≥ 768 px"
    assert w * h <= 1_048_576,    "width × height must be ≤ 1,048,576"
    with open(image_path, "rb") as f:
        resp = requests.post(
            EDIT_URL,
            headers={"api-key": API_KEY},
            data={"model": DEPLOYMENT, "prompt": prompt, "size": size},
            files=[("image", (Path(image_path).name, f, "image/png"))],
            timeout=120,
        )
    resp.raise_for_status()
    return resp.json()["data"][0]["b64_json"]


def show(b64: str, save_as: str | None = None) -> None:
    """Decode base64 PNG, optionally save, and display inline."""
    raw = base64.b64decode(b64)
    if save_as:
        out = OUT_DIR / save_as
        out.write_bytes(raw)
        print(f"Saved → {out}")
    display(IPyImage(data=raw))


## 4. Generate an image from a text prompt

`generate_image` posts a JSON body to `/mai/v1/images/generations`.
The prompt below asks for a clean product shot — a good baseline for editing in section 6.


In [ ]:
b64 = generate_image(
    prompt=(
        "Studio product shot of a cobalt-blue ceramic coffee mug on a white background, "
        "soft-box lighting, sharp focus, minimal drop shadow"
    ),
    width=1024,
    height=1024,
)
show(b64, save_as="01-mug-studio.png")


## 5. Adjust image dimensions

Both dimensions must be ≥ 768 px and their product ≤ 1,048,576.
This allows common aspect ratios without exceeding the pixel budget:

| Ratio | Width | Height | Typical use |
|---|---|---|---|
| 1 : 1 square | 1024 | 1024 | Social thumbnails |
| 4 : 3 landscape | 1024 | 768 | Hero banners, slides |
| 3 : 4 portrait | 768 | 1024 | Mobile, story format |

Run the same prompt in all three ratios to see how the model adapts composition.


In [ ]:
sizes = [
    ("square",    1024, 1024),
    ("landscape", 1024,  768),
    ("portrait",   768, 1024),
]
prompt = "A cozy independent bookshop interior, warm amber light, packed shelves"

for name, w, h in sizes:
    print(f"\n{name.upper()} — {w}×{h}")
    b64 = generate_image(prompt, width=w, height=h)
    show(b64, save_as=f"02-{name}.png")


## 6. Edit an existing image

The edits endpoint (`/mai/v1/images/edits`) accepts:

- `image` — the source PNG as a multipart file upload
- `prompt` — a natural-language instruction describing the change
- `size` — output dimensions as `"WxH"` (same pixel-budget rules apply)

Effective edit prompts name the element to change **and** explicitly state what to keep.
The cell below re-colours the mug from section 4 while preserving lighting and composition.


In [ ]:
b64_edit = edit_image(
    image_path=str(OUT_DIR / "01-mug-studio.png"),
    prompt=(
        "Change the mug colour to terracotta orange; "
        "keep the lighting, background, and composition identical"
    ),
    size="1024x1024",
)
show(b64_edit, save_as="03-mug-edited.png")


## 7. Image Editing

Upload a JPEG or PNG image and describe the targeted change while asking the model to preserve every other detail. Put your image in the /images folder and update line 3 with your image

In [ ]:
import mimetypes

INPUT_IMAGE = 'images/car2.png'  # Replace with your JPEG or PNG file.
EDIT_PROMPT = 'Remove the finger blur mark while preserving every other detail of this photograph.'
input_path = Path(INPUT_IMAGE)
if not input_path.is_file():
    raise FileNotFoundError(f'Input image not found: {input_path}')
mime_type = mimetypes.guess_type(input_path.name)[0] or 'image/png'
if mime_type not in {'image/jpeg', 'image/png'}:
    raise ValueError('Use a JPEG or PNG input image.')

edited_output_path = OUT_DIR / f'{input_path.stem}_edited.png'
edited_b64 = edit_image(str(input_path), EDIT_PROMPT)
show(edited_b64, save_as=edited_output_path.name)
edited_paths = [str(edited_output_path)]

### Before and after examples

Remove an unwanted object from the image - in this case declutter the kitchen scene and light focus on the flowers to enhance

![Before and after: greeting card removed from a flower arrangement](output/flowers_before_after.png)

Remove the glare from screens and glass, remove clutter chairs around the table so that the scene is simple and muted. Wonderful edits for a social post

![Before and after: finger blur removed from a laptop photograph](output/laptop_before_after.png)

### Compare your original and edited images

The next cell loads the original photo and the edited output created above, then displays them side by side at the same preview width. This makes it easier to compare the model's changes even when the source images have different dimensions.

In [ ]:
import ipywidgets as widgets

# Fixed pixel width so both previews render at the same on-screen size regardless of source resolution.
PREVIEW_WIDTH = '480px'

def preview_image(path: str, label: str, image_format: str) -> widgets.VBox:
    return widgets.VBox([
        widgets.Label(label),
        widgets.Image(value=Path(path).read_bytes(), format=image_format, layout=widgets.Layout(width=PREVIEW_WIDTH, height='auto')),
    ])

original_format = 'jpeg' if mime_type == 'image/jpeg' else 'png'
display(widgets.HBox([
    preview_image(str(input_path), 'Original', original_format),
    preview_image(edited_paths[0], 'Edited', 'png'),
], layout=widgets.Layout(justify_content='space-around')))

## 8. Your Turn to Explore

Try adapting the cells above to investigate:

1. **Prompt specificity** — compare `"a mug"` against `"a matte ceramic mug with a brushed
   steel handle"`. How much does material description affect output fidelity?
2. **Iterative editing** — use the output of one edit as the input to the next.
   Does the model preserve composition across multiple rounds?
3. **Aspect ratio × subject** — does a tall subject (a standing person, a tower, a wine bottle)
   produce a more coherent result when the aspect ratio matches the subject orientation?

In [ ]:
# Your experiments here


## 9. Summary

You used MAI-Image-2.5 to:

- Generate a PNG from a text prompt via `/mai/v1/images/generations`
- Work within the pixel-budget constraints across three common aspect ratios
- Edit an existing image with a targeted instruction via `/mai/v1/images/edits`

**Next steps**

- [MAI-Image-2.5-Flash capsule](../mai-image-2.5-flash/) — production-scale batch generation
- [MAI-Image-2.5-Pro capsule](../mai-image-2.5-pro/) — portrait quality and accurate text rendering

**References**

- [Deploy and use MAI image models in Microsoft Foundry](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/how-to/use-foundry-models-mai-image?tabs=python)
- [MAI-Image-2.5 model page](https://microsoft.ai/models/mai-image-2-5/)
- [Image generation primer](../../../docs/primers/image-generation.md)